# S2-PR-01/02/03 — Federated runtime and convergence tools

**Runtime evidence, not model-quality evidence.** This notebook reads public summaries only. It does not open private Parquet, execute Flower, access TEST, or calculate Quality Retention.


## 1. Locate this repository
The code works when started from the repository root or from `notebooks/`.


In [ ]:
from pathlib import Path
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config/federated/fl_runtime_workload.v1.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project repository")

def load_public(relative_path):
    path = ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Run the documented benchmark first: {relative_path}")
    return json.loads(path.read_text(encoding="utf-8"))

scale = load_public("docs/evidence/s2-pr-01/scale_summary.v1.json")
convergence = load_public("docs/evidence/s2-pr-02/convergence_validation.v1.json")
profile = load_public("docs/evidence/s2-pr-03/profile_summary.v1.json")
recommendations = load_public("docs/evidence/s2-pr-03/runtime_recommendations.v1.json")


## 2. Three different client counts
Population is the number of real-user-derived virtual clients registered with Flower. Clients per round is the sampled subset. Concurrency is the cap on simultaneous backend client workers. Do not confuse these counts.


In [ ]:
[{k: row.get(k) for k in ("trial_id", "population", "clients_per_round",
   "concurrency", "status", "warm_round_p50_seconds", "warm_round_p95_seconds",
   "sampled_tree_peak_rss_bytes")} for row in scale["points"]]


## 3. Scale acceptance and repeatability
The chosen point must pass the fixed technical/resource checks and repeat successfully. Dormant virtual clients are not all trained during a six-round run.


In [ ]:
{"scale_status": scale["status"],
 "chosen_population": scale.get("chosen_population"),
 "repeatability": scale.get("repeatability"),
 "limitations": scale.get("limitations", [])}


## 4. Profiling across concurrency
Round 1 is warm-up for timing summaries. Rounds 2–6 are measured. RSS is a sampled sum over the driver and descendants; shared pages may be counted more than once. It is not exact unique physical memory.


In [ ]:
profile["points"]


## 5. Runtime recommendations
Recommendations apply to this workload and this measured hardware only. They are not the final scientific FedAvg protocol.


In [ ]:
recommendations


## 6. Convergence definitions
Self convergence uses 95% of the completed run's best VALIDATION value for three scheduled observations. The confirmation round is the third observation. The R1 target uses the first observed crossing of 90% of a verified matching R1. No interpolation is allowed.

The values below come from hand-worked fixtures, not a claim that the smoke model has scientifically converged.


In [ ]:
{"validation_status": convergence["status"],
 "fixture_cases": convergence["cases"],
 "real_r1_status": convergence["real_r1_status"],
 "actual_communication_status": convergence["actual_communication_status"]}


## 7. Scope and next handoff
This bundle supplies #35 scale evidence, #36 deterministic curve/byte arithmetic, and #37 measured runtime profiles. It does not close #43, implement #50, train the final GRU, change Eid's frozen data/metric decisions, or establish DP/Secure Aggregation guarantees.
